# RO-058 Trajectory optimisation: the local planner, every number in the Note

The local planner (a timed elastic band, TEB), the person the map does not have, the receding-horizon loop, the
smoother and the sparsity count live in `images/bandkit.py`; the robot, the straight drive and the least-time
collocation of the whole drive live in `images/trajkit.py`. Every solve starts from a fixed first guess, so nothing
is random and there is no seed to set. The whole notebook takes a few minutes.


In [1]:
import sys, math
sys.path.insert(0, 'images')
import numpy as np
import bandkit as bk
import trajkit as tk
r2, r3, r4 = (lambda x: round(float(x), 2)), (lambda x: round(float(x), 3)), (lambda x: round(float(x), 4))
R = bk.results()
L, F, W = R['loop'], R['free'], R['wrong']
def closest(P, person):
    seg = np.concatenate([np.linspace(P[k, :2], P[k + 1, :2], 20) for k in range(len(P) - 1)])
    return float(bk.gap(seg[:, 0], seg[:, 1], person).min())

## 1. The global plan, the person and the drive (Overview, Figure 1)

In [2]:
route, length = bk.astar()
print('A* route (cell centres):', route.tolist(), ' length', r3(length), 'm')
print('door pose', tk.GOAL[:2], 'facing', math.degrees(tk.GOAL[2]))
print('person: centre', bk.PERSON, 'radius', bk.PERSON_R, 'm; steps in at', bk.T_PERSON, 's')
print('route x = 2.75 passes', r3(abs(2.75 - bk.PERSON[0])), 'm from the person centre; disk overlap if followed:',
      r3(bk.PERSON_R + tk.R_BODY - abs(2.75 - bk.PERSON[0])), 'm')
c = L[24]
print('robot when the person appears: t', c['t'], 'pose', np.round(c['x'][:2], 3), 'heading', r2(math.degrees(c['x'][2])),
      'deg, speed', r3(c['v']), 'm/s')
print('gap from the robot disk to the person then:', r3(np.hypot(*(bk.PERSON - c['x'][:2])) - bk.PERSON_R - tk.R_BODY), 'm')
xs = np.array([o['x'] for o in L]); vs = np.array([o['v'] for o in L])
print('arrival with the person:', L[-1]['t'], 's; without:', F[-1]['t'], 's; difference', r2(L[-1]['t'] - F[-1]['t']), 's')
print('end pose', np.round(L[-1]['x'][:2], 4), 'heading', r2(math.degrees(L[-1]['x'][2])), 'deg')
print('least gap from the robot disk to the person along the drive:', r3(bk.gap(xs[:, 0], xs[:, 1], bk.PERSON).min()), 'm')
print('lowest speed after the person appears (before the door):', r3(vs[24:60].min()), 'm/s; top heading', r2(np.degrees(xs[:, 2]).max()), 'deg')
print('cycles', len(L), '; poses per band', min(len(o['P']) for o in L), 'to', max(len(o['P']) for o in L),
      '; median time step', r3(np.median(np.concatenate([o['DT'] for o in L[:60]]))), 's')

A* route (cell centres): [[2.25, 1.25], [2.75, 1.75], [2.75, 2.25], [2.75, 2.75], [2.75, 3.25], [3.25, 3.75], [3.75, 3.75]]  length 3.414 m
door pose [3.6 3.5] facing 90.0
person: centre [2.9  2.75] radius 0.2 m; steps in at 2.4 s
route x = 2.75 passes 0.15 m from the person centre; disk overlap if followed: 0.3 m
robot when the person appears: t 2.4 pose [2.495 1.929] heading 69.15 deg, speed 0.595 m/s
gap from the robot disk to the person then: 0.465 m
arrival with the person: 8.0 s; without: 6.7 s; difference 1.3 s
end pose [3.6002 3.5062] heading 88.79 deg
least gap from the robot disk to the person along the drive: 0.14 m
lowest speed after the person appears (before the door): 0.437 m/s; top heading 123.19 deg
cycles 81 ; poses per band 3 to 12 ; median time step 0.294 s


## 2. A hard limit and a soft one (Figure 5)

In [3]:
from scipy.optimize import minimize_scalar
for w in (2, 1000):
    f = lambda v: (0.18 / v) ** 2 + w * max(0, v - 0.55) ** 2
    r = minimize_scalar(f, bounds=(0.3, 1.0), method='bounded', options=dict(xatol=1e-9))
    print('weight', w, ': lowest cost at', r4(r.x), 'm/s; time cost', r4((0.18 / r.x) ** 2), '; penalty', r4(f(r.x) - (0.18 / r.x) ** 2))
print('time cost at 0.6 m/s:', r4((0.18 / 0.6) ** 2), '; penalty at 0.6 m/s, weight 2:', r4(2 * 0.05 ** 2))

weight 2 : lowest cost at 0.6185 m/s; time cost 0.0847 ; penalty 0.0094
weight 1000 : lowest cost at 0.5502 m/s; time cost 0.107 ; penalty 0.0
time cost at 0.6 m/s: 0.09 ; penalty at 0.6 m/s, weight 2: 0.005


## 3. Smoothing one corner, five points, the whole route (Figures 6, 7)

In [4]:
p0, p1, p2 = np.array([2.25, 1.25]), np.array([2.75, 1.75]), np.array([2.75, 2.25])
d2 = p0 - 2 * p1 + p2
c = (p2 - p0) / np.hypot(*(p2 - p0)); n = np.array([-c[1], c[0]])
print('second difference', d2, '; squared length', d2 @ d2, '; sideways direction n', np.round(n, 3), '; n . d2', r3(n @ d2))
cost = lambda d: float(np.sum((d2 - 2 * d * n) ** 2))
print('cost(d) = 0.25 -', r3(4 * n @ d2), 'd + 4 d^2;  cost(0.1) =', r3(cost(0.1)), '; best d', r3(n @ d2 / 2), 'cost', r3(cost(n @ d2 / 2)))
D = np.array([[1, -2, 1, 0, 0], [0, 1, -2, 1, 0], [0, 0, 1, -2, 1]])
print('D^T D for five points:'); print(D.T @ D)
S = R['smooth']
print('route smoothing: cost', r3(S['cost'](np.zeros(5))), '->', r4(S['cost'](S['exact'])), 'm^2; moves', np.round(S['exact'], 4))
print('cost after each projected-gradient step:', [r4(S['cost'](d)) for d in S['hist'][:6]])

second difference [-0.5  0. ] ; squared length 0.25 ; sideways direction n [-0.894  0.447] ; n . d2 0.447
cost(d) = 0.25 - 1.789 d + 4 d^2;  cost(0.1) = 0.111 ; best d 0.224 cost 0.05
D^T D for five points:
[[ 1 -2  1  0  0]
 [-2  5 -4  1  0]
 [ 1 -4  6 -4  1]
 [ 0  1 -4  5 -2]
 [ 0  0  1 -2  1]]
route smoothing: cost 0.75 -> 0.3972 m^2; moves [ 0.1     0.0037  0.0187 -0.1    -0.1   ]
cost after each projected-gradient step: [0.75, 0.5178, 0.4241, 0.3989, 0.3974, 0.3973]


## 4. The timed elastic band worked by hand (Section 5)

In [5]:
c0 = L[0]; Pw, Dw = c0['warm']; P, DT = c0['P'], c0['DT']
print('first cycle: local goal', np.round(c0['goal'], 4), '; warm start', len(Pw), 'poses; solved', len(P), 'poses,', r3(DT.sum()), 's')
print('solved first two poses', np.round(P[:2], 4).tolist(), '; dT1', r4(DT[0]))
d1 = np.hypot(*(P[1, :2] - P[0, :2])); v, w, acc, _, _ = bk.band_terms(P, DT, 0.0, False)
print('distance', r4(d1), '; v1 = d / dT1 =', r4(d1 / DT[0]), '; turn', r4(P[1, 2] - P[0, 2]), 'rad; w1 =', r4(w[0]),
      '; a1 = v1 / dT1 =', r4(acc[0]))
print('wheels', r3(v[0] + w[0] * 0.1), r3(v[0] - w[0] * 0.1), '; band top wheel speed', r3(np.max(np.abs(np.r_[v + w * 0.1, v - w * 0.1]))))
chord = math.degrees(math.atan2(*(P[1, :2] - P[0, :2])[::-1]))
print('solved: chord', r2(chord), 'deg; headings', r2(math.degrees(P[0, 2])), r2(math.degrees(P[1, 2])), '; halfway', r2(math.degrees(P[:2, 2].mean())))
def nh(P):
    d = np.diff(P[:, :2], axis=0)
    return (np.cos(P[:-1, 2]) + np.cos(P[1:, 2])) * d[:, 1] - (np.sin(P[:-1, 2]) + np.sin(P[1:, 2])) * d[:, 0]
print('warm start first two poses', np.round(Pw[:2], 4).tolist(), '; chord', r2(math.degrees(math.atan2(*(Pw[1, :2] - Pw[0, :2])[::-1]))),
      'deg; arc rule e =', r4(nh(Pw)[0]), '; solved band largest |e|', '%.1e' % np.abs(nh(P)).max())
print('cost: warm start', r2(bk.cost(Pw, Dw, 0.0, False, bk.obstacles(None))), '-> solved', r3(c0['cost']))

first cycle: local goal [2.75   2.5429 1.5708] ; warm start 10 poses; solved 11 poses, 3.236 s
solved first two poses [[2.0, 1.0, 0.5236], [2.1054, 1.1087, 1.0774]] ; dT1 0.4831
distance 0.1514 ; v1 = d / dT1 = 0.3134 ; turn 0.5538 rad; w1 = 1.1464 ; a1 = v1 / dT1 = 0.6488
wheels 0.428 0.199 ; band top wheel speed 0.623
solved: chord 45.87 deg; headings 30.0 61.73 ; halfway 45.87
warm start first two poses [[2.0, 1.0, 0.5236], [2.1604, 1.1604, 0.6545]] ; chord 45.0 deg; arc rule e = 0.0883 ; solved band largest |e| 5.9e-05
cost: warm start 21.38 -> solved 1.182


## 5. The cycle the person appears in: TEB's passes, and the sparsity of its problem (Figures 8, 9)

In [6]:
c = L[24]
Pw, Dw = c['warm']
print('warm start', len(Pw), 'poses: robot disk to person', r3(closest(Pw, bk.PERSON)), 'm; solved', len(c['P']), 'poses,',
      r2(c['DT'].sum()), 's: gap', r3(closest(c['P'], bk.PERSON)), 'm')
obs = bk.obstacles(bk.PERSON)
print('cost with TEB weights after each frame:', [r2(bk.cost(h[0], h[1], c['v'], c['final'], obs)) for h in c['hist']])
for n in (len(c['P']), 47):
    M, names = bk.sparsity(n)
    print(n, 'poses:', len(names), 'unknowns,', r3(100 * M.mean()), 'percent of J^T J can be non-zero')

warm start 9 poses: robot disk to person -0.218 m; solved 10 poses, 3.3 s: gap 0.146 m
cost with TEB weights after each frame: [37.71, 24.75, 22.01, 19.04, 15.37, 11.79, 37.18, 17.6, 3.82, 1.97, 1.65, 1.6, 10.06, 1.71, 1.46, 1.42, 1.42, 1.4, 2.18, 1.44, 1.4, 1.39, 1.39, 1.39]
10 poses: 33 unknowns, 46.924 percent of J^T J can be non-zero
47 poses: 181 unknowns, 9.691 percent of J^T J can be non-zero


## 6. Collocation: the trapezoid rule on the straight drive (Section 6, Figure 10)


In [7]:
r = tk.straight_collocation(N=10); h = r['T'] / 10
print('straight drive, least time with 10 intervals: first interval h', r4(h), 's; v', r4(r['v'][0]), '->', r4(r['v'][1]),
      '; trapezoid', r4(h / 2 * r['v'][1]), 'm; 0.5 a h^2', r4(0.5 * 0.5 * h * h), 'm')
print('a band of 10 poses:', 10 * 3 + 9, 'numbers; 31 knot points of the whole drive and its time:', 31 * (4 + 2) + 1, 'numbers')


straight drive, least time with 10 intervals: first interval h 0.6364 s; v 0.0 -> 0.3182 ; trapezoid 0.1013 m; 0.5 a h^2 0.1013 m
a band of 10 poses: 39 numbers; 31 knot points of the whole drive and its time: 187 numbers


## 7. The wrong side, and a search as the first guess (Figure 11)


In [8]:
print('person moved to', bk.PERSON_LEFT, '; cycle t', W['cycle']['t'])
Pw2, Dw2, cw = W['res_w']; Pa2, Da2, ca = W['res_a']
print('warm start: cost', r2(cw), '; gap to person or table', r3(closest(Pw2, bk.PERSON_LEFT)), 'm; rightmost x', r2(Pw2[:, 0].max()))
print('A* on 0.1 m cells:', W['expanded'], 'cells expanded, route', r3(W['route_len']), 'm')
print('search start: cost', r2(ca), '; gap', r3(closest(Pa2, bk.PERSON_LEFT)), 'm; leftmost x', r2(Pa2[:, 0].min()))
from shapely.geometry import Point, Polygon
print('narrowest gap between the person and the table:', r3(Point(bk.PERSON_LEFT).buffer(bk.PERSON_R, quad_segs=256).distance(Polygon(tk.TABLE))), 'm; robot width', 2 * tk.R_BODY, 'm')

person moved to [2.65 2.75] ; cycle t 2.4
warm start: cost 5.82 ; gap to person or table -0.022 m; rightmost x 3.18
A* on 0.1 m cells: 78 cells expanded, route 2.107 m
search start: cost 1.49 ; gap 0.148 m; leftmost x 2.05
narrowest gap between the person and the table: 0.564 m; robot width 0.5 m


## 8. Refine once: the least-time drive from the A* route (Figures 13, 14)


In [9]:
st = tk.stop_and_turn()
tm = tk.cached('time', tk.collocation_time)
print('least time', r4(tm['T']), 's; knot points', len(tm['X']), '; time between them', r3(tm['T'] / (len(tm['X']) - 1)), 's; largest defect', '%.1e' % np.abs(tm['defects']).max())
ts, xs2 = tk.drive_controls(tm['X'], tm['U'], tm['T'])
print('replayed controls end', r4(np.hypot(*(xs2[-1, :2] - tk.GOAL[:2]))), 'm from the door pose; top speed', r4(xs2[:, 3].max()))
print('least gap to the table at the knots', r4(tk.clearance(tm['X'][:, 0], tm['X'][:, 1]).min()))
print('refined once', r2(tm['T']), 's; stop and turn along the route', r2(st['T']), 's; local planner without the person', F[-1]['t'], 's')
print('straight line, rest to rest, ignoring the table:', r3(tk.bang_coast_bang(tk.LINE_LEN)), 's')


least time 6.1809 s; knot points 31 ; time between them 0.206 s; largest defect 3.9e-10
replayed controls end 0.0031 m from the door pose; top speed 0.6
least gap to the table at the knots 0.0
refined once 6.18 s; stop and turn along the route 11.81 s; local planner without the person 6.7 s
straight line, rest to rest, ignoring the table: 6.147 s
